## DESCARGA DE DATOS DE IMPORTACIONES

Vienen de la página del INDEC. Solo bajo los años que voy a necesitar

In [1]:
#En esta celda importo lo que voy a necesitar para la descarga de los datos y para después descomprimirlos y guardarlos en la carpeta correspondiente porque vienen en formato zip.

import time #para que espere entre descargas 
import zipfile #para archivos zip

import requests 

from config import CRUDO, GEO, ANIOS, MODO_MUESTRA #usa cosas de mi archivo config

c:\Users\renat\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
URL_BASE = "https://comex.indec.gob.ar/files/zips/imports_{anio}_Y.zip" #acá dejo un hueco porque después hago un loop que lo integra y necesito bajar datos de 2021 a 2025

CRUDO.mkdir(parents=True, exist_ok=True)

In [3]:
def descargar(url, destino, reintentos=3):  # función para bajar cualquier archivo, con reintentos por si falla
    """Descarga un archivo si todavía no está en disco.

    url: dirección del archivo.
    destino: ruta donde guardarlo.
    Devuelve la ruta del archivo, o None si falló después de los reintentos.
    """
    if destino.exists():  # si ya lo tengo, no lo vuelvo a bajar
        return destino
    for intento in range(reintentos):
        try:
            respuesta = requests.get(url, timeout=60)
            respuesta.raise_for_status()
            destino.write_bytes(respuesta.content)
            time.sleep(1)
            return destino
        except requests.RequestException as error:
            print(f"{destino.name}: intento {intento + 1} falló ({error})")
            time.sleep(2 ** intento)
    return None

In [4]:
fallidos = []
for anio in ANIOS:
    url = URL_BASE.format(anio=anio)
    destino = CRUDO / f"imports_{anio}_Y.zip"
    if descargar(url, destino) is None:
        fallidos.append(anio)
print(f"{len(ANIOS) - len(fallidos)} descargados, {len(fallidos)} fallidos: {fallidos}")

7 descargados, 0 fallidos: []


In [5]:
for anio in ANIOS:
    zip_path = CRUDO / f"imports_{anio}_Y.zip"
    carpeta = CRUDO / f"imports_{anio}"
    if zip_path.exists() and not carpeta.exists():
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(carpeta)
    print(anio, sorted(p.name for p in carpeta.iterdir()))

2019 ['Leame importaciones.docx', 'impoa19.csv', 'itota19.csv']
2020 ['Leame importaciones.docx', 'impo_soja_y_sub_productos_acum_2002_2020.xlsx', 'impoa20.csv', 'itota20.csv']
2021 ['Leame importaciones.docx', 'impo_soja_y_sub_productos_acum_2002_2021.xlsx', 'impoa21.csv', 'itota21.csv']
2022 ['Leame importaciones.docx', 'impo_soja_y_sub_productos_acum_2002_2022.xlsx', 'impoa22.csv', 'itota22.csv']
2023 ['Leame importaciones.docx', 'impo_soja_y_sub_productos_acum_2002_2023.xlsx', 'impoa23.csv', 'itota23.csv']
2024 ['Leame importaciones.docx', 'impo_soja_y_sub_productos_acum_2002_2024.xlsx', 'impoa24.csv', 'itota24.csv']
2025 ['Leame importaciones.docx', 'impo_soja_y_sub_productos_acum_2002_2025.xlsx', 'impoa25.csv', 'itota25.csv']


# Para el mapa

In [6]:
URL_MAPA = (
    "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/"
    "master/geojson/ne_110m_admin_0_countries.geojson"
)
GEO.mkdir(parents=True, exist_ok=True)
descargar(URL_MAPA, GEO / "ne_110m_admin_0_countries.geojson")

WindowsPath('datos/geo/ne_110m_admin_0_countries.geojson')